In [19]:
# ======================================================================
# DUBLIN RENT INDEX DATA (ready to use)
# ======================================================================
# Input:  the national processed RTB rent index file
#         (dataSet/processed_dataset/rent_index_dataset/rent_2021_onwards.csv)
#
# Same steps as the BER, planning and property price notebooks:
#   1. load the rent CSV
#   2. keep only Dublin and add the postal district
#   3. check for columns that are the same thing as the target
#   4. check for columns with more than 50% missing values
#   5. remove constant columns
#   6. check for duplicate rows (this file has no id column)
#   7. put the columns in a clear order
#
# IMPORTANT: these are PUBLISHED AVERAGES, not single tenancies.
# One row = the average monthly rent for one area, in one quarter,
# for one type of home (e.g. 'Two bed', 'Apartment').
#
# Target / main value (Student 3, temporal analysis):
#   avg_rent -> average monthly rent in euro
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [20]:
# Folder paths

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

rent_file = os.path.join(base, 'dataSet', 'processed_dataset', 'rent_index_dataset', 'rent_2021_onwards.csv')

print(rent_file)

C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\dataSet\processed_dataset\rent_index_dataset\rent_2021_onwards.csv


In [21]:
# Loading the rent file

df_rent = pd.read_csv(rent_file)

print('Rows:', df_rent.shape[0], ' Columns:', df_rent.shape[1])
df_rent.head()

Rows: 80292  Columns: 9


,quarter,bedrooms,property_type,location,avg_rent,year,quarter_no,county,location_level
0,2021Q1,All bedrooms,All property types,Carlow,871.87,2021,1,Carlow,county
1,2021Q1,All bedrooms,All property types,Carlow Town,901.80,2021,1,Carlow,local
2,2021Q1,All bedrooms,All property types,"Tullow, Carlow",839.67,2021,1,Carlow,local
3,2021Q1,All bedrooms,All property types,Cavan,673.46,2021,1,Cavan,county
4,2021Q1,All bedrooms,All property types,Cavan Town,712.73,2021,1,Cavan,local


In [22]:
# Which counties are in the file?

df_rent['county'].value_counts()

county
Dublin       27319
Cork          6688
Galway        4000
Kildare       3937
Meath         2987
Limerick      2806
Tipperary     2801
Wicklow       2465
Mayo          2437
Kerry         2187
Wexford       1954
Waterford     1924
Louth         1807
Westmeath     1745
Donegal       1628
Clare         1624
Offaly        1469
Cavan         1414
Sligo         1394
Monaghan      1376
Laois         1368
Roscommon     1353
Carlow        1163
Longford      1116
Leitrim        677
Kilkenny       653
Name: count, dtype: int64

In [23]:
# Keeping only the Dublin rows and saving them as df_dublin_rent

df_dublin_rent = df_rent[df_rent['county'] == 'Dublin'].copy()

print('All of Ireland:', len(df_rent))
print('Dublin only:   ', len(df_dublin_rent))
print('Different Dublin locations:', df_dublin_rent['location'].nunique())

All of Ireland: 80292
Dublin only:    27319
Different Dublin locations: 155


In [24]:
# How are the Dublin locations written? There are 3 kinds:
#   'Dublin'              -> the whole county
#   'Dublin 12'           -> a whole postal district
#   'Crumlin, Dublin 12'  -> a smaller area inside a district
#   'Swords, Dublin'      -> a smaller area with no district number

df_dublin_rent['location'].value_counts().head(20)

location
Dublin                    825
Blackrock, Dublin         455
Drumcondra, Dublin 3      445
Ballsbridge, Dublin 4     444
Ranelagh, Dublin 6        443
Swords, Dublin            437
Lucan, Dublin             430
Rathmines, Dublin 6       413
Terenure, Dublin 6W       401
Clontarf, Dublin 3        396
Dun Laoghaire, Dublin     393
Rathgar, Dublin 6         384
Glasnevin, Dublin 9       384
Balbriggan, Dublin        378
Donnybrook, Dublin 4      374
Rathfarnham, Dublin 16    374
Cabra, Dublin 7           366
Malahide, Dublin          359
Tallaght, Dublin 24       354
Sandymount, Dublin 4      353
Name: count, dtype: int64

In [25]:
# Adding the postal district (same district names as the BER, planning and property data).
#   'Crumlin, Dublin 12' -> the part after the comma -> 'Dublin 12'
#   'Dublin 12'          -> no comma, so the whole name -> 'Dublin 12'

dublin_districts = ['Dublin 1', 'Dublin 2', 'Dublin 3', 'Dublin 4', 'Dublin 5', 'Dublin 6',
                    'Dublin 6W', 'Dublin 7', 'Dublin 8', 'Dublin 9', 'Dublin 10', 'Dublin 11',
                    'Dublin 12', 'Dublin 13', 'Dublin 14', 'Dublin 15', 'Dublin 16', 'Dublin 17',
                    'Dublin 18', 'Dublin 20', 'Dublin 22', 'Dublin 24']

district_part = []
for location in df_dublin_rent['location']:
    if ',' in location:
        district_part.append(location.split(',')[1].strip())
    else:
        district_part.append(location.strip())

df_dublin_rent['dublin_district'] = district_part

In [26]:
# Reading the addresses and Eircodes of the Dublin sales from the property price file.
# We use them to find out which district a place name (e.g. 'Howth') belongs to.

import re

ppr_file = os.path.join(base, 'dataSet', 'processed_dataset', 'property_price_dataset', 'ppr_2021_onwards.csv')

df_ppr = pd.read_csv(ppr_file, low_memory=False)
df_ppr = df_ppr[df_ppr['county'] == 'Dublin']

# address in capital letters, with single spaces
ppr_address = df_ppr['address'].fillna('').str.upper().str.replace(r'\s+', ' ', regex=True)

# district from the Eircode: 'D13...' -> 'Dublin 13' (empty if it is not a D Eircode)
ppr_eircode = df_ppr['eircode'].fillna('').str.upper().str.strip()
eircode_number = ppr_eircode.str.extract(r'^D(\d\d|6W)')[0].str.lstrip('0')
ppr_district = 'Dublin ' + eircode_number

# Eircode starting with A or K = outside the numbered districts (Swords, Blackrock ...)
ppr_outside = ppr_eircode.str[:1].isin(['A', 'K'])

print('Dublin sales read:', len(df_ppr))
print('Sales with a D Eircode:', ppr_district.notnull().sum())

Dublin sales read: 106619
Sales with a D Eircode: 65369


In [27]:
# The place name is the part before the comma: 'Howth, Dublin' -> 'Howth'

place_name = df_dublin_rent['location'].str.split(',').str[0].str.strip()

no_district_yet = ~df_dublin_rent['dublin_district'].isin(dublin_districts)
not_whole_county = df_dublin_rent['location'] != 'Dublin'

places_to_check = sorted(place_name[no_district_yet & not_whole_county].unique())

print('Places to check:', len(places_to_check))
print(places_to_check)

Places to check: 24
['Balbriggan', 'Blackrock', 'Booterstown', 'Dalkey', 'Donabate', 'Dun Laoghaire', 'Glenageary', 'Howth', 'Killiney', 'Kinsealy', 'Lucan', 'Lusk', 'Malahide', 'Monkstown', 'Mount Merrion', 'Portmarnock', 'Rathcoole', 'Rush', 'Saggart', 'Sandycove', 'Shankill', 'Skerries', 'Stillorgan', 'Swords']


In [28]:
# For each place, we look at the sales whose address has that place name
# (as a whole part between commas) and check their Eircodes.
# A place gets a district only if:
#   - at least 10 of those sales have a D Eircode, and
#   - at least 90% of them are in the same district, and
#   - 10% or less have an A or K Eircode (outside the districts).

place_to_district = {}

for place in places_to_check:
    pattern = r'(?:^|,)\s*' + re.escape(place.upper()) + r'\s*(?:,|$)'
    has_place = ppr_address.str.contains(pattern, regex=True)

    districts_found = ppr_district[has_place].dropna()
    n_with_district = len(districts_found)
    n_outside = ppr_outside[has_place].sum()

    decision = 'no district'
    if n_with_district >= 10:
        top_district = districts_found.value_counts().index[0]
        top_share = districts_found.value_counts(normalize=True).iloc[0]
        outside_share = n_outside / (n_outside + n_with_district)
        if top_share >= 0.9 and outside_share <= 0.1:
            place_to_district[place] = top_district
            decision = top_district

    print(place.ljust(15), ' sales with D Eircode:', str(n_with_district).rjust(4),
          ' with A/K Eircode:', str(n_outside).rjust(4), ' ->', decision)

print()
print('Dictionary:', place_to_district)

Balbriggan       sales with D Eircode:    1  with A/K Eircode: 1307  -> no district
Blackrock        sales with D Eircode:    2  with A/K Eircode: 1839  -> no district
Booterstown      sales with D Eircode:    0  with A/K Eircode:  208  -> no district
Dalkey           sales with D Eircode:    0  with A/K Eircode:  489  -> no district
Donabate         sales with D Eircode:    0  with A/K Eircode:  567  -> no district
Dun Laoghaire    sales with D Eircode:    0  with A/K Eircode: 1220  -> no district
Glenageary       sales with D Eircode:    0  with A/K Eircode:  349  -> no district
Howth            sales with D Eircode:  323  with A/K Eircode:    0  -> Dublin 13
Killiney         sales with D Eircode:    3  with A/K Eircode:  562  -> no district
Kinsealy         sales with D Eircode:    9  with A/K Eircode:  310  -> no district
Lucan            sales with D Eircode:    0  with A/K Eircode: 2280  -> no district
Lusk             sales with D Eircode:    0  with A/K Eircode:  436  -> no dis

In [29]:
# Giving the district to the rent rows of those places

filled = 0
for place in place_to_district:
    rows = (place_name == place) & ~df_dublin_rent['dublin_district'].isin(dublin_districts)
    df_dublin_rent.loc[rows, 'dublin_district'] = place_to_district[place]
    filled = filled + rows.sum()

print('Rent rows filled from the dictionary:', filled)

Rent rows filled from the dictionary: 894


In [30]:
# Anything that is not a real district becomes 'Dublin (no district)'
# (e.g. 'Swords, Dublin', 'Blackrock, Dublin').
# The row for the whole county ('Dublin') is not one district, so it gets 'All Dublin'.

not_real = ~df_dublin_rent['dublin_district'].isin(dublin_districts)
df_dublin_rent.loc[not_real, 'dublin_district'] = 'Dublin (no district)'

df_dublin_rent.loc[df_dublin_rent['location'] == 'Dublin', 'dublin_district'] = 'All Dublin'

df_dublin_rent['dublin_district'].value_counts()

dublin_district
Dublin (no district)    4222
Dublin 4                1962
Dublin 15               1943
Dublin 6                1930
Dublin 8                1893
Dublin 3                1808
Dublin 18               1342
Dublin 9                1276
Dublin 24               1224
Dublin 7                1146
Dublin 13               1077
Dublin 16                956
Dublin 12                933
All Dublin               825
Dublin 1                 803
Dublin 14                684
Dublin 2                 606
Dublin 6W                603
Dublin 5                 509
Dublin 11                472
Dublin 22                353
Dublin 10                290
Dublin 20                254
Dublin 17                208
Name: count, dtype: int64

In [31]:
# location_level says what each row describes.
# The file only has 'county' and 'local', so we add 'district' for rows that are
# exactly a whole district ('Dublin 12').

df_dublin_rent.loc[df_dublin_rent['location'].isin(dublin_districts), 'location_level'] = 'district'

df_dublin_rent['location_level'].value_counts()

location_level
local       24865
district     1629
county        825
Name: count, dtype: int64

In [32]:
# The whole-district rows ('Dublin 12') stop after 2021.
# After that, the rent index only publishes the smaller areas ('Crumlin, Dublin 12').
# So a district rent for later years has to be built from its smaller areas.

whole_district_rows = df_dublin_rent[df_dublin_rent['location_level'] == 'district']
whole_district_rows.groupby('quarter').size()

quarter
2021Q1    486
2021Q2    495
2021Q3    494
2021Q4    154
dtype: int64

In [33]:
# Is any column the same thing as the target (avg_rent)?
# Every other column only DESCRIBES the row (when, where, what kind of home).
# None of them is a rent in another form, so there is nothing to remove here.

df_dublin_rent.columns.tolist()

['quarter',
 'bedrooms',
 'property_type',
 'location',
 'avg_rent',
 'year',
 'quarter_no',
 'county',
 'location_level',
 'dublin_district']

In [34]:
# Missing values in each column, as a percentage of all Dublin rows

missing_percent = df_dublin_rent.isnull().sum() / len(df_dublin_rent) * 100

missing_percent.round(2).sort_values(ascending=False)

quarter            0.0
bedrooms           0.0
property_type      0.0
location           0.0
avg_rent           0.0
year               0.0
quarter_no         0.0
county             0.0
location_level     0.0
dublin_district    0.0
dtype: float64

In [35]:
# Removing the columns where more than 50% of the values are missing
# (in this file there should be none - it is a published table)

columns_to_remove = missing_percent[missing_percent > 50].index

print('Columns removed:', len(columns_to_remove))
print(list(columns_to_remove))

df_dublin_rent = df_dublin_rent.drop(columns=columns_to_remove)

Columns removed: 0
[]


In [36]:
# Finding columns that have only ONE value for every row.
# A column that never changes can't help, so we remove it.
# (nunique counts the different values, ignoring missing ones)

constant_columns = []

for column in df_dublin_rent.columns:
    if df_dublin_rent[column].nunique() <= 1:
        constant_columns.append(column)
        print(column, '->', df_dublin_rent[column].unique()[:3])

print('Constant columns:', len(constant_columns))
print('Column names:', constant_columns)

county -> <StringArray>
['Dublin']
Length: 1, dtype: str
Constant columns: 1
Column names: ['county']


In [37]:
# Removing them

print('Columns before:', df_dublin_rent.shape[1])
df_dublin_rent = df_dublin_rent.drop(columns=constant_columns)
print('Columns after: ', df_dublin_rent.shape[1])

Columns before: 10
Columns after:  9


In [38]:
# This file has no id column. One row is identified by 4 things together:
# quarter + location + bedrooms + property_type.
# Checking that no row appears twice.

key_columns = ['quarter', 'location', 'bedrooms', 'property_type']

print('Duplicate rows:', df_dublin_rent.duplicated(subset=key_columns).sum())   # should be 0

Duplicate rows: 0


In [39]:
# Arranging the columns (nothing is removed - only the order changes):
# dates -> location -> categories -> target

date_columns = ['quarter', 'year', 'quarter_no']
location_columns = ['dublin_district', 'location', 'location_level']
categorical_columns = ['bedrooms', 'property_type']
target_columns = ['avg_rent']

wanted_order = date_columns + location_columns + categorical_columns

# 1. columns from our lists, in our order
new_order = []
for column in wanted_order:
    if column in df_dublin_rent.columns:
        new_order.append(column)

# 2. any column we did not list goes here, so it is NOT lost
for column in df_dublin_rent.columns:
    if column not in new_order and column not in target_columns:
        new_order.append(column)
        print('Not in the lists, kept anyway:', column)

# 3. target always at the end
for column in target_columns:
    if column in df_dublin_rent.columns:
        new_order.append(column)

print('Columns before:', df_dublin_rent.shape[1], ' after:', len(new_order))   # always the same
df_dublin_rent = df_dublin_rent[new_order]

df_dublin_rent.head()

Columns before: 9  after: 9


,quarter,year,quarter_no,dublin_district,location,location_level,bedrooms,property_type,avg_rent
62,2021Q1,2021,1,All Dublin,Dublin,county,All bedrooms,All property types,1731.27
63,2021Q1,2021,1,Dublin (no district),"Balbriggan, Dublin",local,All bedrooms,All property types,1368.82
64,2021Q1,2021,1,Dublin (no district),"Blackrock, Dublin",local,All bedrooms,All property types,2122.55
65,2021Q1,2021,1,Dublin (no district),"Booterstown, Dublin",local,All bedrooms,All property types,2193.63
66,2021Q1,2021,1,Dublin (no district),"Dalkey, Dublin",local,All bedrooms,All property types,2405.23


In [40]:
# Quick check of what we have now

print('Rows:', df_dublin_rent.shape[0], ' Columns:', df_dublin_rent.shape[1])
print(list(df_dublin_rent.columns))

Rows: 27319  Columns: 9
['quarter', 'year', 'quarter_no', 'dublin_district', 'location', 'location_level', 'bedrooms', 'property_type', 'avg_rent']
